In [ ]:
import os

from pathlib import Path
os.chdir(Path.cwd().parent)
import time
import copy
import json
import random
import warnings
import numpy as np
from pathlib import Path
from collections import defaultdict
from IPython.core.magic import register_cell_magic

from data.dataloader import load_dataset
from utils.utils import set_random_seed, Usage
from utils.llm import load_model
from utils.metrics import get_metrics_fn
from utils.multiprocess import multiprocess_call
from modules.post_process import get_label_tokenizer, get_logit, prepare_labels, match_drug_fuzz
from TherapyReason.significance import get_significance
from TherapyReason.therapy import get_dl_candidates, get_kg_candidates, get_therapy, rank_dl_candidates
from TherapyReason.overview import process_solution_overview_wrap
from TherapyReason.reasoning import candidate2string, solution2string_with_overview, template_solution_system, \
    post_process_solution_wrapper, candidate_similarity

set_random_seed(42)
dataset = 'mimic3'
task = 'drugrec'
label_key = 'drugs'
datasets, full_set = load_dataset(dataset, task)
test_set = datasets['test']
label_tokenizer = get_label_tokenizer(full_set, label_key)
cond_tokenizer = get_label_tokenizer(full_set, 'conditions')
proc_tokenizer = get_label_tokenizer(full_set, 'procedures')
drug_tokenizer = label_tokenizer
significance = get_significance(dataset, task)
cache_dir = f"./TherapyReason/cache/{dataset}-{task}"
os.makedirs(cache_dir, exist_ok=True)
drug_similarity = json.loads(Path("TherapyReason/cache/llm_drug_pairwise.json").read_text())
print(len(test_set))

warnings.filterwarnings('ignore')


@register_cell_magic
def skip(line, cell): return None


llm = load_model({"model": "DeepSeek-V3", "server": "ant3", "do_sample": False, "temperature": 0.0, "max_attempts": 10})

In [ ]:
# Candidates
sample_dl_candidates = [[] for _ in range(len(test_set))]
sample_kg_candidates = get_kg_candidates(cache_dir, task, test_set, drug_tokenizer, label_tokenizer)

In [ ]:
# Therapy (or Solution)
solution_file = Path(os.path.join(cache_dir, "ablation", f"solution_nodl.jsonl"))
sample_solutions = get_therapy(solution_file, task, significance, label_key, test_set, sample_dl_candidates, sample_kg_candidates)

In [ ]:
# solution overview
process_solution_overview = process_solution_overview_wrap(os.path.join(cache_dir, "ablation", f"solution_nodl_overview", llm.genre), llm)
solution_overview_output = multiprocess_call(process_solution_overview, sample_solutions, n_workers=8, verbose=False)

## solution reasoning

In [ ]:
# 初始化solution reasoning的cache文件（写空文件）
chunk_size = 20

for key, item in enumerate(list(zip(sample_solutions, solution_overview_output))):
    solution_obj, solution_overview = item
    visits = solution_obj['solutions']

    cache_kg_file = Path(os.path.join(cache_dir, "ablation", f"solution_nodl_kg", llm.genre, f"{key}.json"))
    os.makedirs(os.path.dirname(cache_kg_file), exist_ok=True)
    if not cache_kg_file.exists():
        cached = []
        for visit_id, visit in enumerate(visits):
            _cached = []
            for solution_id, solution in enumerate(visit):
                n_chunks = int(np.ceil(len(solution["kg_drugs"]) / chunk_size))
                _cached.append({"kg_drugs": solution["kg_drugs"], "kg_query": [None] * n_chunks, "kg_response": [None] * n_chunks,
                                "kg_prediction": {},"kg_usage": Usage().dict()})
            cached.append(_cached)

        cache_kg_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")
        print(f"Init {cache_kg_file}")

In [ ]:
# 对KG进行reasoning
template_solution_kg = """\
# Input
## Patient's Context
{solution_string}

## Possible Drugs
{candidate_dict}

# Instructions
Evaluate Possible Drugs. These drugs are obtained from a wide-spread knowledge graph and might be loosely related to the patient.
Consider the relevance of each drug (the keys) to the patient's medical conditions, and predict scores from 0 to 4 in json format:
```json
{{
  "drug name": {{
    "reason": "your reasoning",
    "predict": [0-4]
  }},
  ...
}}
```

# Output Format
Evaluate Possible Drugs: [Your output in json]
"""

def process_solutions_kg(key, item, **kwargs):
    solution_obj, solution_overview = item
    visits = solution_obj['solutions']

    cache_file = Path(os.path.join(cache_dir, "ablation", f"solution_nodl_kg", llm.genre, f"{key}.json"))
    cached = json.loads(cache_file.read_text(encoding="utf-8"))
    for visit_id, visit in enumerate(visits):
        for solution_id, solution in enumerate(visit):
            overview = solution_overview[visit_id][solution_id]['overview']

            n_chunks = int(np.ceil(len(solution['kg_drugs']) / chunk_size).item())
            for i in range(n_chunks):
                if cached[visit_id][solution_id]['kg_response'][i] is not None: continue

                time.sleep(random.randint(0, 4))
                print(f"Patient: {key}\tVisit: {visit_id + 1}/{len(visits)}\tSolution: {solution_id + 1}/{len(visit)}\tKG Chunk: {i}/{n_chunks}")
                kg_candidates = solution["kg_drugs"][i * chunk_size: (i + 1) * chunk_size]
                candidate_string_dict = candidate2string(kg_candidates, solution)
                assert candidate_string_dict

                post_process_solution = post_process_solution_wrapper(drug_tokenizer, candidates=list(candidate_string_dict))
                query = template_solution_kg.format(solution_string=solution2string_with_overview(solution, overview, overview_only=True),
                                                    candidate_dict=json.dumps(candidate_string_dict, indent=2))
                _, (response, prediction), _, usage = llm.understand(query, system=template_solution_system, post_process=post_process_solution, **kwargs)
                cached[visit_id][solution_id]['kg_query'][i] = query
                cached[visit_id][solution_id]['kg_response'][i] = response
                cached[visit_id][solution_id]['kg_prediction'].update(prediction)
                cached[visit_id][solution_id]['kg_usage'] = (usage + cached[visit_id][solution_id]['kg_usage']).dict()
                cache_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")

    return key, cached


sample_kg_reasoning = multiprocess_call(process_solutions_kg, list(zip(sample_solutions, solution_overview_output)), n_workers=16, verbose=False)

# eval

In [ ]:
# 评估DL+KG的结果
def get_dlkg_metric(offset, threshold, kg_reasoning, agg='mean'):
    y_true_all, y_prob_all = [], []
    for i in range(len(test_set)):
        patient_prediction = defaultdict(list)
        for visit_id, visit in enumerate(kg_reasoning[i]):
            for solution_id in range(len(visit)):
                for drug, value in kg_reasoning[i][visit_id][solution_id]['kg_prediction'].items():
                    patient_prediction[drug].append(value['predict'])

        patient_prediction = {drug: np.mean(scores) if agg == 'mean' else np.max(scores) for drug, scores in patient_prediction.items()}
        patient_prediction = [drug for drug, score in patient_prediction.items() if score >= threshold]
        y_true_all.append(prepare_labels(task, [test_set[i][label_key]], label_tokenizer)[0].numpy())
        y_prob_all.append(get_logit(task, list(patient_prediction), label_tokenizer))

    metrics = get_metrics_fn(task)(np.stack(y_true_all), np.array(y_prob_all))
    print(offset, threshold, agg, metrics["precision_samples"], metrics["recall_samples"], metrics["f1_samples"], metrics["jaccard_samples"], metrics['hamming_loss'], metrics['rank'])

for agg in ['mean', 'max']:
    for offset in range(2, 4):
        for threshold in range(offset + 1, min(offset + 5, 5)):
            get_dlkg_metric(offset, threshold, sample_kg_reasoning, agg=agg)